# Reto proyecto: Generación de imágenes de productos e-commerce con `diffusers`

**Alumno/a:** _Escribe aquí tu nombre y apellidos_  
**Objetivo:** utilizar la biblioteca `diffusers` de HuggingFace para generar imágenes de productos para una tienda online mediante Stable Diffusion, experimentar con prompts, parámetros y schedulers, y aplicar la técnica **image-to-image** para crear variaciones de una imagen base.

Este notebook cubre los requisitos del enunciado:

1. Configuración del entorno con `diffusers`, `torch`, `transformers`, `accelerate`, `safetensors` y `PIL`.
2. Detección de GPU si está disponible.
3. Uso del modelo `stabilityai/stable-diffusion-2-1-base`.
4. Configuración de `EulerAncestralDiscreteScheduler`.
5. Generación de imágenes de producto para e-commerce usando prompts y `negative_prompt`.
6. Experimentación con varios prompts, parámetros y schedulers.
7. Generación de variaciones con **image-to-image**.
8. Guardado de todas las imágenes en un directorio de salida.

## 1. Instalación de librerías

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo. En algunos entornos locales quizá ya tengas instaladas estas dependencias.

In [ ]:
# Si usas Google Colab o un entorno nuevo, descomenta y ejecuta esta línea.
# !pip install -q diffusers transformers accelerate safetensors torch pillow matplotlib


## 2. Imports, configuración de dispositivo y directorio de salida

Se comprueba si hay GPU disponible. Si no la hay, el notebook puede ejecutarse en CPU, aunque la generación será mucho más lenta. Para evitar problemas de memoria, el código usa `float16` cuando hay CUDA y activa `attention_slicing`.

In [ ]:
import os
import gc
from pathlib import Path

import torch
from PIL import Image, ImageDraw
from IPython.display import display

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
)

try:
    from diffusers.utils import make_image_grid
except Exception:
    make_image_grid = None

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_ID = "stabilityai/stable-diffusion-2-1-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Dispositivo detectado: {DEVICE}")
print(f"Tipo de datos usado: {DTYPE}")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")


## 3. Funciones auxiliares

Estas funciones organizan el guardado de imágenes, la creación de grids y la limpieza de memoria. La función `build_grid` usa `make_image_grid` de `diffusers` cuando está disponible y, si no, usa una implementación equivalente con PIL.

In [ ]:
def clean_memory():
    """Libera memoria entre generaciones."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def build_grid(images, rows, cols, resize_to=None):
    """Crea una cuadrícula con una lista de imágenes PIL."""
    processed = []
    for img in images:
        img = img.convert("RGB")
        if resize_to is not None:
            img = img.resize(resize_to)
        processed.append(img)

    if make_image_grid is not None:
        return make_image_grid(processed, rows=rows, cols=cols)

    width, height = processed[0].size
    grid = Image.new("RGB", size=(cols * width, rows * height), color="white")
    for i, img in enumerate(processed):
        x = (i % cols) * width
        y = (i // cols) * height
        grid.paste(img, (x, y))
    return grid


def save_image(image, filename):
    """Guarda una imagen PIL en la carpeta outputs y devuelve la ruta."""
    path = OUTPUT_DIR / filename
    image.save(path)
    print(f"Guardado: {path}")
    return path


def create_fallback_product_base(path=OUTPUT_DIR / "base_product_placeholder.png"):
    """Crea una imagen base simple por si no se ha generado aún ninguna imagen previa."""
    img = Image.new("RGB", (512, 512), "white")
    draw = ImageDraw.Draw(img)
    # Dibujo simple tipo zapatilla deportiva, útil como base local para image-to-image.
    draw.rounded_rectangle((95, 260, 405, 340), radius=38, fill=(230, 230, 230), outline=(30, 30, 30), width=4)
    draw.polygon([(120, 260), (210, 185), (315, 260)], fill=(245, 245, 245), outline=(30, 30, 30))
    draw.line((185, 235, 295, 235), fill=(30, 30, 30), width=3)
    draw.line((200, 220, 285, 220), fill=(30, 30, 30), width=3)
    draw.ellipse((340, 290, 380, 330), fill=(200, 200, 200), outline=(30, 30, 30), width=3)
    draw.text((130, 380), "Base image - EcoRun sneaker", fill=(0, 0, 0))
    img.save(path)
    return path


## 4. Cargar pipeline de Stable Diffusion con Euler Ancestral Scheduler

Se usa el modelo solicitado en el enunciado: `stabilityai/stable-diffusion-2-1-base`. El scheduler principal es `EulerAncestralDiscreteScheduler`.

In [ ]:
def load_text2img_pipeline(model_id=MODEL_ID):
    """Carga el pipeline de texto a imagen con EulerAncestralDiscreteScheduler."""
    scheduler = EulerAncestralDiscreteScheduler.from_pretrained(
        model_id,
        subfolder="scheduler"
    )

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=DTYPE,
        safety_checker=None,       # Se desactiva para evitar dependencias extra en algunos entornos educativos.
        requires_safety_checker=False,
    )

    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()

    # xformers es opcional. Si no está instalado, el notebook continúa funcionando.
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("xformers activado para mejorar eficiencia de memoria.")
    except Exception:
        print("xformers no disponible; se usará attention_slicing.")

    return pipe

pipe = load_text2img_pipeline()


## 5. Generación inicial de producto ficticio para e-commerce

Producto elegido: **zapatillas deportivas sostenibles EcoRun Air**.  
La imagen se plantea como fotografía de producto para una tienda online: fondo blanco, iluminación de estudio, alta resolución, aspecto realista y sin logos.

In [ ]:
product_name = "EcoRun Air - zapatillas deportivas sostenibles"

base_prompt = (
    "A high-resolution e-commerce product photo of modern sustainable running sneakers, "
    "white and green colorway, recycled knit fabric texture, clean white studio background, "
    "softbox lighting, realistic product photography, sharp focus, 4k, centered composition"
)

negative_prompt = (
    "low quality, blurry, distorted, deformed, bad proportions, extra shoes, extra objects, "
    "text, watermark, logo, brand name, dirty background, cropped product, unrealistic, noise"
)

generator = torch.Generator(device=DEVICE).manual_seed(1234)

image = pipe(
    prompt=base_prompt,
    negative_prompt=negative_prompt,
    width=512,
    height=512,
    num_inference_steps=35,
    guidance_scale=8.0,
    generator=generator,
).images[0]

save_image(image, "01_ecorun_air_prompt_inicial.png")
display(image)
clean_memory()


## 6. Experimentación con distintos prompts

Se generan varias imágenes del mismo producto con cambios de estilo, ángulo, materiales y presentación. Esto ayuda a comparar qué prompts funcionan mejor para una tienda online.

In [ ]:
prompt_experiments = [
    {
        "name": "white_background_side_view",
        "prompt": (
            "A professional side view product photo of EcoRun Air sustainable running sneakers, "
            "white background, green recycled mesh, premium e-commerce catalog style, realistic lighting"
        ),
        "steps": 35,
        "guidance": 8.0,
        "seed": 11,
    },
    {
        "name": "top_view_minimal",
        "prompt": (
            "A top view product photo of modern eco-friendly running sneakers, minimalist e-commerce layout, "
            "white background, clean shadows, recycled fabric texture, high detail"
        ),
        "steps": 35,
        "guidance": 7.5,
        "seed": 22,
    },
    {
        "name": "premium_lifestyle",
        "prompt": (
            "A premium lifestyle product photo of sustainable running sneakers on a light concrete surface, "
            "soft natural studio lighting, modern ecommerce campaign, realistic, high detail"
        ),
        "steps": 40,
        "guidance": 8.5,
        "seed": 33,
    },
    {
        "name": "black_green_variant",
        "prompt": (
            "A high-resolution e-commerce product photo of black and neon green running sneakers, "
            "recycled technical fabric, futuristic sole, white studio background, sharp realistic product photography"
        ),
        "steps": 40,
        "guidance": 9.0,
        "seed": 44,
    },
]

experiment_images = []

for item in prompt_experiments:
    gen = torch.Generator(device=DEVICE).manual_seed(item["seed"])
    img = pipe(
        prompt=item["prompt"],
        negative_prompt=negative_prompt,
        width=512,
        height=512,
        num_inference_steps=item["steps"],
        guidance_scale=item["guidance"],
        generator=gen,
    ).images[0]
    save_image(img, f"02_prompt_experiment_{item['name']}.png")
    experiment_images.append(img)
    clean_memory()

grid_prompts = build_grid(experiment_images, rows=2, cols=2)
save_image(grid_prompts, "02_grid_experimentacion_prompts.png")
display(grid_prompts)


### Análisis de la experimentación con prompts

- El prompt con **fondo blanco y vista lateral** es el más adecuado para una ficha de producto estándar.
- El prompt **top view minimal** permite mostrar la silueta y el diseño de la zapatilla desde otro ángulo.
- El prompt **premium lifestyle** es más útil para banners o campañas de marketing, aunque se aleja del fondo blanco típico de catálogo.
- El prompt **black green variant** muestra cómo el mismo producto puede adaptarse a una variante de color.

El `negative_prompt` evita resultados no deseados como baja calidad, logos, marcas, texto incrustado, deformaciones o productos recortados.

## 7. Comparación de parámetros: `guidance_scale` y `num_inference_steps`

Aquí se comparan diferentes configuraciones para observar cómo influyen en la calidad, coherencia y fidelidad al prompt.

In [ ]:
parameter_tests = [
    {"name": "low_guidance_fast", "guidance": 5.5, "steps": 25, "seed": 101},
    {"name": "balanced", "guidance": 7.5, "steps": 35, "seed": 102},
    {"name": "high_guidance", "guidance": 10.0, "steps": 35, "seed": 103},
    {"name": "more_steps", "guidance": 8.0, "steps": 50, "seed": 104},
]

param_images = []
param_prompt = (
    "A studio product photo of EcoRun Air sustainable running shoes, white background, "
    "clean e-commerce photography, recycled mesh texture, professional lighting, realistic"
)

for item in parameter_tests:
    gen = torch.Generator(device=DEVICE).manual_seed(item["seed"])
    img = pipe(
        prompt=param_prompt,
        negative_prompt=negative_prompt,
        width=512,
        height=512,
        num_inference_steps=item["steps"],
        guidance_scale=item["guidance"],
        generator=gen,
    ).images[0]
    save_image(img, f"03_parametros_{item['name']}_gs{item['guidance']}_steps{item['steps']}.png")
    param_images.append(img)
    clean_memory()

grid_params = build_grid(param_images, rows=2, cols=2)
save_image(grid_params, "03_grid_comparacion_parametros.png")
display(grid_params)


### Análisis de parámetros

- Un `guidance_scale` demasiado bajo puede generar imágenes más libres, pero menos fieles al prompt.
- Un `guidance_scale` medio, alrededor de 7.5 u 8, suele ofrecer equilibrio entre creatividad y control.
- Un `guidance_scale` muy alto fuerza demasiado el prompt y puede producir artefactos.
- Aumentar `num_inference_steps` puede mejorar detalles, aunque incrementa el tiempo de ejecución.

## 8. Comparación de schedulers

Además del scheduler obligatorio `EulerAncestralDiscreteScheduler`, se prueba `DPMSolverMultistepScheduler` para comparar resultados y demostrar experimentación.

In [ ]:
def generate_with_scheduler(scheduler_name, scheduler, seed):
    pipe.scheduler = scheduler
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    img = pipe(
        prompt=param_prompt,
        negative_prompt=negative_prompt,
        width=512,
        height=512,
        num_inference_steps=35,
        guidance_scale=8.0,
        generator=gen,
    ).images[0]
    save_image(img, f"04_scheduler_{scheduler_name}.png")
    clean_memory()
    return img

scheduler_images = []

# Scheduler 1: Euler Ancestral
scheduler_euler = EulerAncestralDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
scheduler_images.append(generate_with_scheduler("euler_ancestral", scheduler_euler, seed=202))

# Scheduler 2: DPM Solver Multistep
scheduler_dpm = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
scheduler_images.append(generate_with_scheduler("dpm_solver_multistep", scheduler_dpm, seed=202))

scheduler_grid = build_grid(scheduler_images, rows=1, cols=2)
save_image(scheduler_grid, "04_grid_comparacion_schedulers.png")
display(scheduler_grid)

# Se restaura Euler para las siguientes secciones.
pipe.scheduler = EulerAncestralDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")


### Análisis de schedulers

- `EulerAncestralDiscreteScheduler` suele generar variaciones creativas y con buen detalle.
- `DPMSolverMultistepScheduler` puede ser eficiente y estable con un número moderado de pasos.
- La comparación permite escoger el scheduler más adecuado según el equilibrio buscado entre calidad, coherencia y tiempo de generación.

## 9. Image-to-image: variaciones de una imagen base

En esta sección se utiliza una imagen base para generar variaciones del producto con cambios guiados por texto. Se parte preferentemente de una imagen generada en las secciones anteriores. Si no existe, se crea una imagen base simple con PIL para que la sección pueda ejecutarse igualmente.

La técnica **image-to-image** conserva parte de la estructura de la imagen inicial y aplica cambios según el prompt. El parámetro `strength` controla cuánto se transforma la imagen original.

In [ ]:
# Liberamos el pipeline texto-a-imagen antes de cargar image-to-image para ahorrar memoria.
del pipe
clean_memory()

base_image_path = OUTPUT_DIR / "02_prompt_experiment_white_background_side_view.png"
if not base_image_path.exists():
    base_image_path = create_fallback_product_base()

init_image = Image.open(base_image_path).convert("RGB").resize((512, 512))
print(f"Imagen base usada para image-to-image: {base_image_path}")
display(init_image)

img2img_scheduler = EulerAncestralDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    scheduler=img2img_scheduler,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
img2img_pipe = img2img_pipe.to(DEVICE)
img2img_pipe.enable_attention_slicing()

try:
    img2img_pipe.enable_xformers_memory_efficient_attention()
    print("xformers activado para image-to-image.")
except Exception:
    print("xformers no disponible para image-to-image; se usará attention_slicing.")


In [ ]:
img2img_variations = [
    {
        "name": "navy_orange",
        "prompt": (
            "A high-resolution e-commerce product photo of the same running sneakers, "
            "navy blue upper with orange accents, white studio background, realistic product photography"
        ),
        "strength": 0.45,
        "guidance": 8.0,
        "seed": 301,
    },
    {
        "name": "white_gold",
        "prompt": (
            "A premium e-commerce product photo of the same running sneakers, white upper with subtle gold details, "
            "clean studio background, luxury sports product photography"
        ),
        "strength": 0.50,
        "guidance": 8.5,
        "seed": 302,
    },
    {
        "name": "trail_version",
        "prompt": (
            "A product photo of the same sneakers redesigned as trail running shoes, rugged sole, forest green details, "
            "white ecommerce background, realistic, high detail"
        ),
        "strength": 0.55,
        "guidance": 8.0,
        "seed": 303,
    },
]

variation_images = []

for item in img2img_variations:
    gen = torch.Generator(device=DEVICE).manual_seed(item["seed"])
    result = img2img_pipe(
        prompt=item["prompt"],
        negative_prompt=negative_prompt,
        image=init_image,      # En versiones antiguas de diffusers este parámetro podía llamarse init_image.
        strength=item["strength"],
        guidance_scale=item["guidance"],
        num_inference_steps=35,
        generator=gen,
    ).images[0]
    save_image(result, f"05_img2img_variation_{item['name']}.png")
    variation_images.append(result)
    clean_memory()

img2img_grid = build_grid([init_image] + variation_images, rows=1, cols=4, resize_to=(384, 384))
save_image(img2img_grid, "05_grid_image_to_image_base_y_variaciones.png")
display(img2img_grid)


### Análisis de image-to-image

- La imagen base mantiene la composición general del producto.
- Con `strength=0.45` se conserva más la estructura original y se aplican cambios moderados.
- Con `strength=0.55` la variación es más visible, útil para crear versiones nuevas como una zapatilla de trail.
- El `negative_prompt` evita que aparezcan logos, texto, deformaciones o elementos extra que no encajan con una ficha de e-commerce.

## 10. Resumen final del proyecto

En este proyecto se ha desarrollado una solución de generación de imágenes de producto para e-commerce con HuggingFace `diffusers`:

- Se ha configurado el entorno con `diffusers`, `torch`, `PIL` y dependencias habituales.
- Se ha usado GPU si está disponible mediante `torch.cuda.is_available()`.
- Se ha usado el modelo `stabilityai/stable-diffusion-2-1-base`.
- Se ha configurado el scheduler `EulerAncestralDiscreteScheduler`.
- Se han generado imágenes de un producto ficticio: **zapatillas EcoRun Air**.
- Se han usado prompts positivos y `negative_prompt` para mejorar la calidad.
- Se han comparado prompts, parámetros y schedulers.
- Se ha aplicado **image-to-image** para crear variaciones de una imagen base.
- Se han guardado los resultados en la carpeta `outputs` con nombres descriptivos.

### Conclusión

La generación de imágenes con Stable Diffusion puede acelerar el proceso creativo en una tienda online, permitiendo explorar variantes visuales de un producto sin realizar sesiones fotográficas físicas desde el inicio. No obstante, para uso real en producción sería recomendable aplicar revisión humana, control de calidad, coherencia visual entre productos y comprobación de que las imágenes no contengan logos, textos o elementos engañosos.

## 11. Celdas opcionales para comprimir resultados

Estas celdas crean un archivo `.zip` con las imágenes generadas. Solo ejecútalas después de haber generado las imágenes.

In [ ]:
# Opcional: comprimir la carpeta outputs en un ZIP.
# Ejecuta esta celda al final si quieres descargar todas las imágenes generadas.

import zipfile

zip_path = Path("outputs_generados_diffusers_ecommerce.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_DIR.glob("*.png"):
        zipf.write(file_path, arcname=file_path.name)

print(f"ZIP creado: {zip_path.resolve()}")
